In [23]:
# Import libraries

import pandas as pd, numpy as np, sqlalchemy as sqla, shutil, decouple
from sqlalchemy import String, Integer, Numeric
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.2f}".format)

engine = sqla.create_engine(decouple.config("MIS_DB"), pool_pre_ping=True)
red = sqla.create_engine(decouple.config("MIS_DASHBOARD"), pool_pre_ping=True)
mis_file_path = decouple.config("MIS_FILE_PATH")

In [24]:
# Sell-Out SQL Query 

    #   SELECT 
    #         sout.`year`, sout.`month`, ""Sell-Out"" AS `type`, ref_ad.sales_group, sout.account_name, 
    #         ref_ad.account_chain, ref_sd.store_name, sout.store_code, ref_sd.city, ref_sd.province, 
    #         ref_sd.region, ref_sd.bpc_region, ref_l.lead_name, sout.item_desc, sout.item_code, 
    #         ref_ic.product_code, ref_pd.product_name, ref_pd.brand, ref_pd.packaging, ref_pd.volume,
    #         ref_pd.product_class, ref_pd.`usage`, ref_pd.product_type, ref_pd.product_category,
    #         ref_pd.target_type, sout.qty, sout.amount, sout.net_amount
    #     FROM sales.sout_prtl AS sout

    #     -- Add account details
    #     LEFT JOIN ref.account_details AS ref_ad
    #         ON sout.account_name = ref_ad.account_name

    #     -- Add store details
    #     LEFT JOIN ref.store_details AS ref_sd
    #         ON sout.account_name = ref_sd.account_name
    #         AND sout.store_code = ref_sd.store_code

    #     -- Add assigned team leader
    #     LEFT JOIN ref.lead_names AS ref_l
    #         ON ref_ad.lead_id = ref_l.lead_id
    #         AND sout.`year` = ref_l.`year`
    #         AND sout.`month` = ref_l.`month`

    #     -- Add item codes
    #     LEFT JOIN ref.item_codes AS ref_ic
    #         ON ref_ad.account_chain = ref_ic.account_chain
    #         AND sout.item_code = ref_ic.item_code
            
    #     -- Add product details
    #     LEFT JOIN ref.product_details AS ref_pd
    #         ON ref_ic.product_code = ref_pd.product_code1
            
    #     WHERE  sout.`year` > 2024
    #     ORDER BY sout.account_name;

In [25]:
# Pull data from Database

sout_prtl = pd.read_sql("SELECT * from staging.sout_prtl_mtd", engine)
ref_ad = pd.read_sql("SELECT * from ref.account_details", engine)
ref_sd = pd.read_sql("SELECT * from ref.store_details", engine)
ref_l = pd.read_sql("SELECT * from ref.lead_names", engine)
ref_ic = pd.read_sql("SELECT * from ref.item_codes", engine)
ref_pd = pd.read_sql("SELECT * from ref.product_details", engine)

# Change data types of JOIN columns

ref_ic["item_code"] = ref_ic["item_code"].astype("string")
sout_prtl["item_code"] = sout_prtl["item_code"].astype("string")
sout_prtl["net_amount"] = sout_prtl["net_amount"].astype("float64")
ref_l["year"] = ref_l["year"].astype("int64")

In [26]:
sout_prtl_net_mt = sout_prtl["net_amount"].sum()
sout_prtl_net_mt

np.float64(41044859.922199994)

In [27]:
# JOINS

sout_ytd = pd.merge(sout_prtl, ref_ad, how="left", on="account_name")
sout_ytd = pd.merge(sout_ytd, ref_sd, how="left", on=["account_name","store_code"])
sout_ytd = pd.merge(sout_ytd, ref_l, how="left", on=["lead_id","year","month"])
sout_ytd = pd.merge(sout_ytd, ref_ic, how="left", on=["account_chain","item_code"])
sout_ytd["product_code"] = sout_ytd["product_code"].str.strip()
sout_ytd = pd.merge(sout_ytd, ref_pd, how="left", left_on="product_code", right_on="product_code1")



sout_ytd.head()

,year,month,account_name,store_name_x,store_code,item_desc_x,item_code,qty,amount,net_amount,...,product_name,common_name,brand,packaging,volume,product_class,usage,product_type,product_category,target_type
0,2026,June,Philippine Seven Corporation,Philippine Seven Corporation B.F. Homes,2,i-White Skin Whitening Aqua Moisturizing,80200276,1.00,25.00,18.75,...,iWhite Korea Aqua Moisturizer Whitening Vita -...,iWhite Korea Aqua Moisturizer Whitening Vita S...,iWhite Korea,Sachet,6ml,Skin Care,Daily,Moisturizer,Regular,Baseline
1,2026,June,Philippine Seven Corporation,Philippine Seven Corporation B.F. Homes,2,iWhite Moisturizing Facial Wash 10ml,80200706,1.00,23.00,17.25,...,iWhite Korea Facial Wash Whitening Vita - 10ml...,iWhite Korea Facial Wash Whitening Vita Sachet,iWhite Korea,Sachet,10ml,Skin Care,Daily,Facial Wash,Regular,Baseline
2,2026,June,Philippine Seven Corporation,Philippine Seven Corporation B.F. Homes,2,iWhite Korea Aqua Moisturizer Glow 6ml,80201143,1.00,27.00,20.25,...,iWhite Korea Aqua Moisturizer Glow Power Brigh...,iWhite Korea Aqua Moisturizer Glow Power Brigh...,iWhite Korea,Sachet,6ml,Skin Care,Daily,Moisturizer,Regular,Baseline
3,2026,June,Philippine Seven Corporation,Philippine Seven Corporation B.F. Homes,2,iWhite Korea Glow Up Whip Wash 10ml,80201144,1.00,24.00,18.00,...,iWhite Korea Glow-Up Whip Wash - 10ml (PFWS),iWhite Korea Glow-Up Whip Wash Sachet,iWhite Korea,Sachet,10ml,Skin Care,Daily,Facial Wash,Regular,Baseline
4,2026,June,Philippine Seven Corporation,Philippine Seven Corporation B.F. Homes,2,iWhite Korea Glow Up Whip Wash 10ml,80201144,1.00,24.00,18.00,...,iWhite Korea Glow-Up Whip Wash - 10ml (PFWS),iWhite Korea Glow-Up Whip Wash Sachet,iWhite Korea,Sachet,10ml,Skin Care,Daily,Facial Wash,Regular,Baseline


In [28]:
# Sum of net_amount after initial JOIN

sout_ytd.head(1)

,year,month,account_name,store_name_x,store_code,item_desc_x,item_code,qty,amount,net_amount,...,product_name,common_name,brand,packaging,volume,product_class,usage,product_type,product_category,target_type
0,2026,June,Philippine Seven Corporation,Philippine Seven Corporation B.F. Homes,2,i-White Skin Whitening Aqua Moisturizing,80200276,1.00,25.00,18.75,...,iWhite Korea Aqua Moisturizer Whitening Vita -...,iWhite Korea Aqua Moisturizer Whitening Vita S...,iWhite Korea,Sachet,6ml,Skin Care,Daily,Moisturizer,Regular,Baseline


In [29]:
# Column name list of reqested report

# Year	Month	Sales Group	Account Name	Chain Name	
# Account Type	Lead Name	Product Code	Product Name	
# Class	Brand	Type	Net Amount # qty in price, branch

In [30]:
# Removing _y columns and renaming _x duplicated columns

cols_to_drop = [c for c in sout_ytd.columns if c.endswith('_y')]
sout_ytd = sout_ytd.drop(columns=cols_to_drop)
sout_ytd = sout_ytd.rename(columns=lambda x: x.replace('_x', '')) 
sout_ytd.head(1)

# Selecting columns and renaming
sout_ytd= sout_ytd[["year", "month","sales_group","account_name","account_chain", "account_type","lead_name", 
                    "product_code","main_code","common_name","product_class","brand","product_type","net_amount"]]

sout_ytd_dashboard = sout_ytd.rename(columns={
    "common_name":"product_name"})


# sout_ytd = sout_ytd.rename(columns={
#     "year":"Year","month":"Month","sales_group":"Sales Group","account_name":"Account Name","account_chain":"Account Chain","account_type":"Account Type","lead_name":"Lead Name",
#     "main_code":"Main Code","common_name":"Product Name", "product_class":"Class","brand":"Brand","product_type":"Type","net_amount":"Net Amount","main_code":"Main code","qty":"Qty in pieces", "store_name":"Store name"})

sout_ytd_dashboard = sout_ytd_dashboard[sout_ytd_dashboard["year"] > 2023]
sout_ytd_dashboard.head(1)

,year,month,sales_group,account_name,account_chain,account_type,lead_name,product_code,main_code,product_name,product_class,brand,product_type,net_amount
0,2026,June,S2,Philippine Seven Corporation,Philippine Seven Corporation,Outright,Rubylie Fabrigas,BACS,BACS,iWhite Korea Aqua Moisturizer Whitening Vita S...,Skin Care,iWhite Korea,Moisturizer,18.75


In [31]:
# sout_ytd["net_amount"].dtype

In [32]:
    #         sout.`year`, sout.`month`, ""Sell-Out"" AS `type`, ref_ad.sales_group, sout.account_name, 
    #         ref_ad.account_chain, ref_sd.store_name, sout.store_code, ref_sd.city, ref_sd.province, 
    #         ref_sd.region, ref_sd.bpc_region, ref_l.lead_name, sout.item_desc, sout.item_code, 
    #         ref_ic.product_code, ref_pd.product_name, ref_pd.brand, ref_pd.packaging, ref_pd.volume,
    #         ref_pd.product_class, ref_pd.`usage`, ref_pd.product_type, ref_pd.product_category,
    #         ref_pd.target_type, sout.qty, sout.amount, sout.net_amount

In [33]:
# sout_ytd_dashboard.shape

In [34]:
sout_checking = sout_ytd_dashboard[sout_ytd_dashboard[["year", "month","sales_group","account_name","account_chain","account_type","lead_name", "product_code","main_code","product_name","product_class","brand","product_type","net_amount"]].isna().any(axis=1)]

sout_checking

,year,month,sales_group,account_name,account_chain,account_type,lead_name,product_code,main_code,product_name,product_class,brand,product_type,net_amount


In [35]:
sout_checking.to_excel(r"C:\anwell copies\checker\dashboard_checker\sellout_dashboard_checker.xlsx", index=False)

In [36]:
## Sell out summary columns

sout_ytd_dashboard = sout_ytd_dashboard[["year", "month", "sales_group", "account_name", "lead_name", "product_code", "net_amount"]]
sout_ytd_dashboard.columns



Index(['year', 'month', 'sales_group', 'account_name', 'lead_name',
       'product_code', 'net_amount'],
      dtype='str')

In [37]:
# Group by year, month, sales_group, account_name, lead_name, product_code, net_amount
sout_ytd_dashboard = sout_ytd_dashboard.groupby(["year", "month", "sales_group", 
"account_name", "lead_name", "product_code"]).agg({"net_amount":"sum"}).reset_index()


In [38]:
sout_ytd_dashboard.head()

,year,month,sales_group,account_name,lead_name,product_code,net_amount
0,2026,June,S1,Watsons Personal Care Store (Phils.) Inc.,Jeric Aquino,AAWB,1666.00
1,2026,June,S1,Watsons Personal Care Store (Phils.) Inc.,Jeric Aquino,ABBLS,680.40
2,2026,June,S1,Watsons Personal Care Store (Phils.) Inc.,Jeric Aquino,ABBLT,67.20
3,2026,June,S1,Watsons Personal Care Store (Phils.) Inc.,Jeric Aquino,ABBNS,302.40
4,2026,June,S1,Watsons Personal Care Store (Phils.) Inc.,Jeric Aquino,ABBNT,67.20


In [ ]:
# # Try catch  for duplicates


# df_to_insert = sout_ytd_dashboard

# try:
#     existing = pd.read_sql('SELECT * FROM `dashboard-sales`.sell_out_staging', con=red)
#     merged = df_to_insert.merge(existing, on=list(df_to_insert.columns), how="left", indicator=True)
#     new_rows = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
# except Exception:
#     new_rows = df_to_insert

# print(f"New rows to insert: {len(new_rows)}")

# new_rows.to_sql(
#     name="sell_out",
#     con=red,
#     schema="dashboard-sales",
#     if_exists="append",
#     index=False,
#     chunksize=1000,
#     method="multi",
# )


New rows to insert: 247215


247215

In [74]:
null_net = df_to_insert[df_to_insert["net_amount"].isna()]
print(len(null_net))
display(null_net.head(20))


0


,year,month,sales_group,account_name,account_chain,account_type,lead_name,product_code,main_code,product_name,product_class,brand,product_type,net_amount


In [75]:
# # Enforce NOT NULL with no default on every sell_out_staging column
# # (dtype= on to_sql only shapes CREATE TABLE; it doesn't alter an existing table)

# alter_sql = """
# ALTER TABLE `dashboard-sales`.`sell_out_staging`
#     MODIFY `year`          INT           NOT NULL,
#     MODIFY `month`         VARCHAR(20)   NOT NULL,
#     MODIFY `sales_group`   VARCHAR(255)  NOT NULL,
#     MODIFY `account_name`  VARCHAR(255)  NOT NULL,
#     MODIFY `account_chain` VARCHAR(255)  NOT NULL,
#     MODIFY `account_type`  VARCHAR(255)  NOT NULL,
#     MODIFY `lead_name`     VARCHAR(255)  NOT NULL,
#     MODIFY `product_code`  VARCHAR(255)  NOT NULL,
#     MODIFY `main_code`     VARCHAR(255)  NOT NULL,
#     MODIFY `product_name`  VARCHAR(255)  NOT NULL,
#     MODIFY `product_class` VARCHAR(255)  NOT NULL,
#     MODIFY `brand`         VARCHAR(255)  NOT NULL,
#     MODIFY `product_type`  VARCHAR(255)  NOT NULL,
#     MODIFY `net_amount`    DECIMAL(15,5) NOT NULL
# """

# with red.begin() as conn:
#     conn.execute(sqla.text(alter_sql))


In [76]:
# #tosqlfile
# df_to_insert = sout_ytd_dashboard.rename(columns={
#     "Product Name": "product_name",
#     "Class": "class",
# })

# try:
#     existing = pd.read_sql('SELECT * FROM `dashboard-sales`.sell_out_staging', con=red)
#     merged = df_to_insert.merge(existing, on=list(df_to_insert.columns), how="left", indicator=True)
#     new_rows = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
# except Exception:
#     new_rows = df_to_insert

# print(f"New rows to insert: {len(new_rows)}")

# # Export to SQL file instead of inserting directly
# sql_path = r"C:\anwell copies\files\sell_out_staging.sql"

# cols = list(new_rows.columns)
# col_str = ", ".join(f"`{c}`" for c in cols)

# with open(sql_path, "w", encoding="utf-8") as f:
#     f.write(f"-- sell_out_staging INSERT statements\n")
#     f.write(f"-- Total rows: {len(new_rows)}\n\n")
#     for _, row in new_rows.iterrows():
#         values = []
#         for val in row:
#             if pd.isna(val):
#                 values.append("NULL")
#             elif isinstance(val, (int, float)):
#                 values.append(str(val))
#             else:
#                 values.append("'" + str(val).replace("'", "''") + "'")
#         f.write(f"INSERT INTO `dashboard-sales`.`sell_out_staging` ({col_str}) VALUES ({', '.join(values)});\n")

# print(f"SQL file saved: {sql_path}")


In [77]:
# #tocsv
# # Export new_rows to CSV for LOAD DATA LOCAL INFILE
# csv_path = r"C:\anwell copies\files\sell_out_staging.csv"
# new_rows.to_csv(csv_path, index=False)
# print(f"CSV saved: {csv_path}")
# print(f"Rows: {len(new_rows)}")


In [78]:
# # Dataframe for sell-out report per branch
# sout_ytd_per_branch = sout_ytd.groupby([
#     "Year",
#     "Month",
#     "Sales Group",
#     "Account Name",
#     "Account Chain",
#     "Account Type",
#     "Lead Name",
#     "Main code",
#     "Product Name",
#     "Class",
#     "Brand",
#     "Type"
# ]).agg({
#     "Net Amount": "sum",
#     "Qty in pieces": "max"
# }).reset_index()

# sout_ytd_per_branch.head(15)

In [79]:
# sout_ytd_per_branch["Net Amount"].sum()